# Topical Lectures, Controls
## Real drone, part 4: the parameters of your drone

Andreas Freise, 31.10.2026

### Goal

Before we fly the drone with our own loop, we need to know the inner loops of the drone better. This notebook measures how fast the drone follows a velocity setpoint, with which delay, and how noisy its estimate is while it hovers. The result is a set of parameters that describe the behaviour of your drone: two numbers for the velocity loop and two bounds from the noise. With them, a model of the drone can predict what our loop will do before it flies; that model runs on the other laptops of the group, see below.

### Loops, setpoints and feedback

Every loop has the same shape. It compares a measurement with the value it should have, the **setpoint**; the difference is the **error signal**, and from it the controller computes the **feedback signal**. What the feedback acts on depends on the loop. In the innermost loop of the firmware it drives the motors. In every loop above, the feedback sets the setpoint of the next loop inside. The drone flies with such a cascade: the feedback of our position loop on the laptop sets the velocity setpoint of the firmware's velocity loop, whose feedback sets the tilt setpoint of the attitude loop, whose feedback drives the motors.

Seen from our loop, the drone with all its inner loops is the system we control: our feedback signal, a velocity, goes in, and a measured position comes out. This notebook measures how that system responds, how fast the measured velocity follows a step of its setpoint and with which delay. It is the response of a loop that someone else tuned, and we need its two numbers before we can choose our gains.

The names are the same in the text, the formulas and the code. `x` and `vx` are the measured position and velocity along an axis, the estimate the drone sends by radio. `x_target`, $x_0$ in the formulas, is the setpoint of our position loop, the value the target jumps to by `STEP`. The feedback signal of our loop is a velocity; in the code it is `vx_fb`, and it is limited to `v_max`.

What you should be able to do afterwards:

* **Read the response time** of a loop you did not build off its step response, by eye and with a fit.
* **Name the two numbers** that describe how the drone follows a velocity setpoint, `tau` and the dead time, and say what each does to our position loop.
One flight, about twenty seconds in the air: the drone takes off, hovers, receives a velocity setpoint of 0.2 m/s for 1.5 s and lands. Milestone 3 closes the notebook: `TAU`, `DELAY`, `KP_MAX` and `KD_MAX` recorded.

### Tools for the rest of the group

The drone laptop is busy with the flights; everything around them runs on other laptops with the analysis notebooks. `drone_analysis_01_flight_plan` flies a model of the drone with any gains in a second. With typical parameters it tells which gains are worth flying; from Milestone 3 on, with the parameters recorded here, it predicts each flight before it is flown. `drone_analysis_02_summary_figure` works on a copy of the `data/` folder of this laptop and makes the table and the figure for the summary.

### Safety

* Fly only in the marked area. One person at the keyboard, one watching the drone, nobody within arm's length while the motors run.
* The person at the keyboard keeps the cursor in the input box. **Enter** aborts and lands. The stop button of the cell also lands; press it once only, a second interrupt cuts the landing short. After a landing the box is still open: press Enter once to close it, nothing else happens.
* The flight lands by itself when the drone leaves a 1 m box around the start, when no state arrives any more, when the battery drops below 3.1 V under load, or when the position estimate becomes unreliable (flow sensor blocked or broken).
* After a tumble or crash: run the status cell; if it says locked, the cell after it clears the lock, otherwise switch the drone off and on.
* Full battery. Below 3.5 V at rest the drone lacks thrust and the response is not the one of the course.


In [ ]:
# Importing packages
import numpy as np
import matplotlib.pyplot as plt
import module
import crazyflie

%matplotlib inline

# The label of your drone as it reads on the sticker:
# LABEL = 2, or LABEL = 'A' for a letter.
LABEL = None
URI = crazyflie.drone_uri(LABEL)

## Test of the abort

Before the first flight we make sure that Enter stops a flight. A dummy loop uses the same listener as the flights.

Run this once per session. The loop runs for 10 s; an input box appears (in VS Code at the top of the window, in JupyterLab below the cell). Press Enter in it and the loop must report the abort. If it does not, do not fly from this program.


In [ ]:
crazyflie.bench_test_abort()

## Status before take-off

Ready to fly means battery, flow deck and supervisor in order. The cell reads the status over the radio; run it before every flight.

Battery above 3.5 V, flow deck attached, not locked is what we want to see. `is locked: yes` means the supervisor has locked the motors after a tumble, a crash or a hard landing, which happens more often on a hard floor: the cell after this one clears the lock and shows the status again. If it stays locked, switch the drone off and on.


In [ ]:
with crazyflie.connect(URI) as scf:
    crazyflie.status(scf)

In [ ]:
# Only when the status says locked: clear the lock, then check again.
with crazyflie.connect(URI) as scf:
    crazyflie.recover(scf)
    crazyflie.status(scf)

## The results of the hand test

The hand test gave the sign of the gains and the noise of the estimate with the motors off. Copy the recorded values in; the noise is compared with the hover below.



In [ ]:
# Copy your results of the hand test here:
KP_SIGN = None     # +1 or -1, from the hand test
NOISE_X = None     # m, standard deviation of the position estimate while the drone lay still
NOISE_V = None     # m/s, standard deviation of the velocity estimate

## Flight 1: how fast does the drone follow a velocity setpoint?

This flight measures the system we control: how fast the measured velocity follows a step of its setpoint, and with which delay. One flight under the controllers of the firmware, one step of the velocity setpoint in the air, the velocity recorded.

Put the drone flat on the floor in the flying area, with its forward side (`+x`, between the motors M1 and M4) pointing into the free space and the two blue LEDs at the back towards you. The drone takes off, hovers, and after 1.5 s receives the velocity setpoint 0.2 m/s along `x` for 1.5 s, then 0 again, then lands. It moves about 30 cm. No loop of ours is closed; this is the firmware alone.

When the cell starts, the drone stays on the ground for about two seconds: the estimator is reset and watched for 1.5 s to make sure the flow deck delivers, then the drone arms and takes off. In the air it hovers for 2 s under the firmware before the handover (`hover_s`). The printout gives the elapsed time at each stage.

The cell ends with the landing. When the drone is down and the printout says `landed`, press **Enter** once in the input box to close it. After a landing that is all Enter does.


In [ ]:
run_id = crazyflie.identify_response(step=0.2, step_s=1.5, uri=URI)
name_id = crazyflie.save(run_id, 'identify')

## Task: read the response off the plot

The two numbers by eye first, before any fit does it for us.

The cell plots the measured velocity against the time since the setpoint step, with the setpoint and its 63 % line. Before you run the fit, read two numbers off the plot and record them in the cell below: the dead time, between the step and the moment the velocity starts to rise, and the response time `tau`, from that moment to the 63 % line. A first order response has no overshoot; if the velocity overshoots or oscillates, note that too.


In [ ]:
vs = crazyflie.velocity_step(run_id)
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(vs['t'], vs['v_meas'], '.-', label='measured vx')
ax.axhline(vs['v_set'], color='0.5', lw=0.8, label='setpoint')
ax.axhline(0.63 * vs['v_set'], color='0.8', lw=0.8, ls=':', label='63 % of the setpoint')
ax.set_xlabel('time since the setpoint step [s]')
ax.set_ylabel('velocity [m/s]')
ax.set_xticks(np.arange(0, vs['t'][-1] + 0.1, 0.2))
ax.legend(loc='lower right')
ax.grid()


In [ ]:
# Record what you read off the plot:
TAU_BY_EYE = None      # s, from the start of the rise to the 63 % line
DELAY_BY_EYE = None    # s, from the setpoint step to the start of the rise

## The response time of the velocity loop

Now the fit gives the same two numbers, and we compare.

The measured velocity after the setpoint step is fitted with a first order response with a dead time,

$$v(t) = v_{fb}\,\bigl(1 - e^{-(t - t_d)/\tau}\bigr) \quad \text{for } t > t_d .$$

`tau` is the time in which the velocity reaches 63 percent of the setpoint; `t_d` is the delay before anything happens (radio, estimator, attitude loop). The fit is a grid search, nothing more. Compare it with the two numbers you read off the plot, then record the two numbers of the fit below. If the measured velocity overshoots or oscillates, the inner loops are not well described by one time constant, and the design below will be optimistic.


In [ ]:
fit = crazyflie.fit_tau(run_id)
tau, delay = fit['tau_s'], fit['delay_s']     # the fit; record them by hand in the next cell

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(fit['t'], fit['v_meas'], label='measured vx')
ax.plot(fit['t'], fit['v_fit'], 'k--', label=f"first order fit, tau = {tau:.2f} s, delay {fit['delay_s']:.2f} s")
ax.axhline(fit['v_set'], color='0.5', lw=0.8, label='setpoint')
ax.axhline(0.63 * fit['v_set'], color='0.8', lw=0.8, ls=':')
ax.set_xlabel('time since the setpoint step [s]')
ax.set_ylabel('velocity [m/s]')
ax.legend(loc='lower right')
ax.grid()


In [ ]:
# Record the parameters of the velocity loop:
TAU = None       # s, the response time of the velocity loop, from the fit
DELAY = None     # s, the dead time, from the fit

## The noise of the estimate

The noise of the estimate in flight, and the bounds it puts on the gains, come from the hover before the setpoint step, where the velocity setpoint was zero.

The gains multiply the noise of the position estimate onto the feedback, and that sets a third limit on them. The drone hovered for 1.5 s with a zero velocity setpoint, so the spread of the position and the velocity in those records is the noise of the estimate, plus a little drift. The cell prints it next to the noise measured with the motors off, and turns it into upper bounds on `|Kp|` and `|Kd|`, such that the noise alone moves the feedback by less than 0.05 m/s. Record the two bounds below; every setting of the gains is compared with them before it flies. On a drone with a good flow deck they are far away.


In [ ]:
r_id = crazyflie.load_run(name_id)                 # the identification flight as arrays
pid = r_id['pid']
hold = pid['step'] < 0.5                           # the records before the setpoint step: the hover
std_x, std_y = np.std(pid['x'][hold]), np.std(pid['y'][hold])
std_vx, std_vy = np.std(pid['vx'][hold]), np.std(pid['vy'][hold])
print(f"hover before the step: std x {1000 * std_x:.1f} mm, y {1000 * std_y:.1f} mm, "
      f"vx {1000 * std_vx:.0f} mm/s, vy {1000 * std_vy:.0f} mm/s ({hold.sum()} records)")
print(f"motors off, hand test:    std position {1000 * NOISE_X:.1f} mm, velocity {1000 * NOISE_V:.0f} mm/s")

# the gains multiply the noise onto the feedback: |Kp| times the position noise, |Kd| times the velocity noise
jitter = 0.05                                      # m/s, the jitter of the feedback we accept from the noise alone
print(f"noise bounds: |Kp| < {jitter / max(std_x, std_y):.1f} 1/s, |Kd| < {jitter / max(std_vx, std_vy):.1f}")


In [ ]:
# Record the bounds from the noise:
KP_MAX = None    # 1/s, upper bound on |Kp| from the position noise
KD_MAX = None    # upper bound on |Kd| from the velocity noise

### Milestone 3: the parameters of your drone

The parameters that describe your drone are complete here; every prediction of the model drone rests on them.

Four values are recorded now: `TAU` and `DELAY`, the response to a velocity setpoint, and `KP_MAX` and `KD_MAX`, the noise. Together with `KP_SIGN` they are what the tuning starts from, on the drone laptop and in the model of the drone alike. Passed when the fit followed the measured velocity, the dead time is below about 0.3 s, and the bounds are far from the gains you intend to fly.


## Further reading: the formulas behind the model

Our controller computes the feedback, a velocity,

$$v_{fb} = K_p\,(x - x_0) + K_d\,\dot x + K_i \int (x - x_0)\,dt ,$$

with the sign convention of the simulator (error = value minus setpoint, so the gains are negative). The velocity loop of the firmware follows that feedback with the time constant `TAU` measured above, $\tau\,\ddot x = v_{fb} - \dot x$. With $K_i = 0$ the two together give

$$\tau\,\ddot x + (1 - K_d)\,\dot x - K_p\,(x - x_0) = 0 ,$$

a damped oscillator with

$$\omega_0 = \sqrt{-K_p/\tau}, \qquad \zeta = \frac{1 - K_d}{2\,\omega_0\,\tau}.$$

This is the lesson of the morning in two formulas. $K_p$ sets the natural frequency, so the speed of the response; $K_d$ changes the damping. The velocity loop damps the position loop by itself, through the term $\dot x/\tau$: with $K_d = 0$ the damping is $\zeta = 1/(2\,\omega_0\,\tau)$, above one for every $K_p$ you will start with. $K_d$ therefore does little until $K_p$ is large, and a positive $K_d$ removes damping. The integrator has no place in the oscillator formulas: it pushes the feedback until the mean error is zero, slowly, and what it accumulates during the step it has to shed afterwards, as an overshoot.

Three things in the real loop are not in the formulas:

1. **The limit of the feedback.** The feedback is clamped at `v_max`. At the moment of the step the controller asks for $K_p \cdot \text{step}$; once that exceeds `v_max`, the first part of the response is a flight at `v_max`, the same for all gains, and a larger gain does not make the response faster. The gains then only decide how the drone arrives at the target.
2. **The dead time.** The feedback acts $t_d$ later. With growing $K_p$ the dead time first adds overshoot, then oscillation, then the loop runs away. This is what limits the gain of every real loop; the model drone includes it.
3. **The noise of the sensor.** The gains multiply the noise of the estimate onto the feedback; the hover gave the bounds `KP_MAX` and `KD_MAX` above.


## If something goes wrong

The messages and symptoms seen in flight, and what to do about each.

* `No drone answers at ...`: the drone is off, or not on the channel of its label.
* `is locked: yes` after a tumble, a crash or a hard landing: run the recover cell under the status cell; if it stays locked, switch the drone off and on.
* `No flow deck detected`: switch off, reseat the deck, switch on.
* The drone wanders off during the hold although the feedback is small: the floor has too little texture for the flow sensor. Fly over the mat, or check what the sensor sees while you move the drone by hand over the floor: `with crazyflie.connect(URI) as scf: crazyflie.flow_check(scf)` in a cell.
* `state samples stale`: the radio link dropped. Check the Crazyradio and the address, keep the laptop close to the flying area.
* The loop never prints `handover`: the take-off failed, usually a low battery or a locked supervisor.
* `a flow deck that was not delivering`, `does not follow the height sensor`, `runs away while the drone lies still`, `after the take-off` or `height sensor frozen`: the deck stopped delivering, which happens without any other sign, also right after switching on. Switch the drone off and on, connect again.
